In [2]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [3]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [10]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

GENRE_ROOT = os.path.join(DATA_ROOT, "genres_stems")

GENRES = sorted(os.listdir(GENRE_ROOT))
STEMS = {
    'drums': 'drums.wav',
    'vocals': 'vocals.wav',
    'bass': 'bass.wav',
    'other': 'other.wav'
}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [20]:
print(GENRES)

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


In [49]:
def build_dataset(root_dir, val_split=0.17, seed=42):

    # Root should point to genres_stems
    genre_root = os.path.join(root_dir, "genres_stems")

    train_dataset = {g: {k: [] for k in STEMS} for g in GENRES}
    val_dataset   = {g: {k: [] for k in STEMS} for g in GENRES}

    rng = random.Random(seed)

    lower_threshold = 4 * 1024
    upper_threshold = 5.0491 * 1024 * 1024

    for genre in GENRES:

        genre_path = os.path.join(genre_root, genre)

        if not os.path.isdir(genre_path):
            continue

        valid_songs = []

        for song in os.listdir(genre_path):

            song_path = os.path.join(genre_path, song)

            if not os.path.isdir(song_path):
                continue

            corrupted = False

            for stem_key, stem_file in STEMS.items():

                stem_path = os.path.join(song_path, stem_file)

                # Completeness check
                if not os.path.isfile(stem_path):
                    corrupted = True
                    break

                size = os.path.getsize(stem_path)

                # Corruption check
                if size < lower_threshold:
                    corrupted = True
                    break

            if not corrupted:
                valid_songs.append(song)

        # Stratified shuffle split (within genre)
        rng.shuffle(valid_songs)

        split_idx = int(len(valid_songs) * (1 - val_split))

        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        # Helper to populate dictionary
        def add_to_dict(target_dict, song_list):
            for song in song_list:
                for stem_key, stem_file in STEMS.items():
                    stem_path = os.path.join(genre_path, song, stem_file)
                    target_dict[genre][stem_key].append(stem_path)

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset

In [43]:
currupted_songs = 0
total_songs_lt_five = 0
stem_count = 0
total_song_gt_fivepzerofourninethree = 0

threshold_corrupted = 4 * 1024
threshold_total = 5.0491 * 1024 * 1024  # MB → bytes
threshold_total_gt = 5.0493 * 1024 * 1024


for genre in GENRES:
    genre_path = os.path.join(GENRE_ROOT, genre)
  
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)

        for stem in os.listdir(song_path):
            stem_path = os.path.join(song_path, stem)
            stem_count += 1
            file_size = os.path.getsize(stem_path)

            if file_size < threshold_corrupted:
                currupted_songs += 1

            if file_size < threshold_total:
                total_songs_lt_five += 1

            if file_size > threshold_total_gt:
                total_song_gt_fivepzerofourninethree += 1
                

print(f'total songs {stem_count}')
print(f'courrupted song {currupted_songs}')
print(f'total songs less than 5.0491 {total_songs_lt_five}')
print(f'total song greater than 5.0493 {total_song_gt_fivepzerofourninethree}')
print(f'answer of question 1 is {currupted_songs + total_songs_lt_five}')
print(f'answer of question 2 is {abs(total_song_gt_fivepzerofourninethree-total_songs_lt_five)}')

total songs 4000
courrupted song 0
total songs less than 5.0491 1256
total song greater than 5.0493 184
answer of question 1 is 1256
answer of question 2 is 1072


In [45]:
tr, val = build_dataset(DATA_ROOT)

In [50]:
train_reggae_drums = len(tr['reggae']['drums'])
val_country_vocals = len(val['country']['vocals'])

print("Training reggae drums:", train_reggae_drums)
print("Validation country vocals:", val_country_vocals)
print("Absolute difference:",
      abs(train_reggae_drums - val_country_vocals))

Training reggae drums: 34
Validation country vocals: 17
Absolute difference: 17


In [51]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: {genre: {stem: [paths...]}}
    Output:
        df: DataFrame of files having silence >= threshold_sec
    """

    records = []
    total_files = 0

    for genre, stem_dict in dataset_dict.items():
        for stem_name, file_list in stem_dict.items():

            for file_path in file_list:

                total_files += 1

                # Load audio
                y, sr = librosa.load(file_path, sr=sr)

                total_duration = librosa.get_duration(y=y, sr=sr)

                # Find non-silent intervals
                intervals = librosa.effects.split(y, top_db=top_db)

                silence_type = []
                max_silence = 0

                # CASE A: Fully silent
                if len(intervals) == 0:
                    max_silence = total_duration
                    silence_type.append("FULL")

                else:
                    # Convert intervals to seconds
                    intervals_sec = intervals / sr

                    # CASE B: Start silence
                    start_silence = intervals_sec[0][0]
                    if start_silence > 0:
                        max_silence = max(max_silence, start_silence)
                        silence_type.append("START")

                    # CASE C: End silence
                    end_silence = total_duration - intervals_sec[-1][1]
                    if end_silence > 0:
                        max_silence = max(max_silence, end_silence)
                        silence_type.append("END")

                    # CASE D: Middle silence
                    for i in range(1, len(intervals_sec)):
                        gap = intervals_sec[i][0] - intervals_sec[i-1][1]
                        if gap > 0:
                            max_silence = max(max_silence, gap)
                            silence_type.append("MIDDLE")

                # Store result
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(set(silence_type)),
                        "File_Path": file_path
                    })

    print("Total files scanned:", total_files)

    df = pd.DataFrame(records)
    return df

In [53]:
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
print(len(df_silence))

Total files scanned: 2264
564


In [55]:
combined = {}

for genre in GENRES:
    combined[genre] = {}
    for stem in STEMS:
        combined[genre][stem] = tr[genre][stem] + val[genre][stem]

df_silence_full = find_long_silences(combined, threshold_sec=DURATION, top_db=TOP_DB)

vocals_silence_count = df_silence_full[df_silence_full["Stem"] == "vocals"].shape[0]

print(vocals_silence_count)

Total files scanned: 2744
310


In [56]:
vocals_df = df_silence_full[df_silence_full["Stem"] == "vocals"]

avg_silence = vocals_df["Max_Silence_Sec"].mean()

print(round(avg_silence, 2))

13.19


In [58]:
jazz_drums_silence_train = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums")
].shape[0]

print(jazz_drums_silence_train)

20


In [65]:
jazz_drums_middle_train = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"].str.contains("MIDDLE"))
].shape[0]

print(jazz_drums_middle_train)

20


In [66]:
jazz_drums_long_silence = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
].shape[0]

print(jazz_drums_long_silence)

7


In [67]:
rock_path = os.path.join(GENRE_ROOT, "rock")
rock_songs = sorted(os.listdir(rock_path))
first_song = rock_songs[0]

print(first_song)

rock.00000


In [69]:
song_path = os.path.join(rock_path, first_song)

drums, _ = librosa.load(os.path.join(song_path, "drums.wav"), sr=SR)
vocals, _ = librosa.load(os.path.join(song_path, "vocals.wav"), sr=SR)
bass, _   = librosa.load(os.path.join(song_path, "bass.wav"), sr=SR)
other, _  = librosa.load(os.path.join(song_path, "other.wav"), sr=SR)

mix = mix[:int(SR * DURATION)]
print(len(mix))

110250


In [70]:
import numpy as np

rms = np.sqrt(np.mean(mix**2))
print(round(rms, 2))

0.2


In [71]:
rms = librosa.feature.rms(y=mix)[0].mean()
print(round(rms, 2))

0.19


In [72]:
normalized_mix = mix / np.max(np.abs(mix))

In [73]:
max_val = np.max(normalized_mix)
print(round(max_val, 2))

0.99
